# Cenário 1 | ZGLMM214 | EMPRESA 4008 (não centro)

| Propriedade | Definição |
|---|---|
| Objeto | `dev_procurement.corp_curated.tbl_ds_pro_zglmm214_cadastro_fornecedor` |
| Ambiente / objeto | DEV / tabela, somente leitura |
| DESCRIBE | `DESCRIBE_ZGLMM214.csv`: 19 colunas, tipos STRING, BOOLEAN, TIMESTAMP e DATE |
| Clustering | Não informado no DESCRIBE |
| Campo de centro | **Não existe**; `cod_empresa` é recorte de empresa, nunca centro |
| SAP | Empresa **4008**; Fornecedor **em branco**; Grupo de contas **em branco**; Layout apenas apresentação |
| Filtro DL | `cod_empresa = '4008'` em todas as consultas de dados |
| Volume esperado | **183.266** linhas, medido no HTML exploratório; revalidar na execução |
| Chave observada no universo | `cod_fornecedor + cod_empresa`: 780.854 combinações para 780.854 linhas, 0 grupos repetidos |

### Alertas
| Severidade | Observação |
|---|---|
| 🔴 ALTA | Dados pessoais e de contato: nunca exportar valor literal; apenas hash ou contagem. |
| 🟡 MÉDIA | Tabela não aplica filtros SAP automaticamente; somente Empresa 4008 está definida neste cenário. |
| 🟡 MÉDIA | Fornecedores excluídos e sem e-mail permanecem na fonte. Não filtrá-los sem confirmar o mesmo comportamento no SAP. |
| ⚪ BAIXA | Não houve coluna 100% sem valor nem duplicidade idêntica no universo explorado; confirmar no recorte. |

**Como usar:** importar no Databricks e executar Run All. Cada célula é independente. O nome do arquivo contém `Centro4008` exclusivamente para seguir a convenção legada de nomes; **4008 é empresa, não centro**. Não comparar `dateingest` com período de negócio. Fonte: HTML exploratório executado e CSV do DESCRIBE.

### Índice
0 Recorte; 1 Metadados; 2 Volumetria; 3 Filtro empresa; 4 Granularidade; 5 Duplicidade; 6 Preenchimento; 7 Cardinalidade; 8 Domínio; 9 Indicadores; 10 Numéricos; 11 Totais; 12 Datas; 13 Códigos; 14 Checksum; 15 Amostra; 16 Distribuição; 17 Freshness; 18 Regras ZGLMM214; 19 Amostra ampliada; 20 Resumo; 21 Extração completa.

## 0. Recorte SAP × Datalake

Registra explicitamente que o código 4008 é empresa e que os demais campos da tela SAP ficam vazios.

In [0]:
SELECT 'ZGLMM214' AS transacao, '4008' AS empresa, 'cod_empresa' AS campo_datalake, 'Fornecedor e Grupo de contas em branco' AS demais_filtros_sap;

## 1. Metadados

Conferir comentários e propriedades da tabela.

In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`;

In [0]:
DESCRIBE DETAIL `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`;

In [0]:
DESCRIBE HISTORY `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`;

## 2. Volumetria

Mede o recorte e compara com o volume da exploratória, sem pressupor estabilidade do snapshot.

In [0]:
SELECT COUNT(*) AS linhas_atuais, 183266 AS linhas_esperadas_exploratoria, COUNT(DISTINCT `cod_fornecedor`) AS fornecedores_distintos, COUNT(DISTINCT `cod_grupo_contas_fornecedor`) AS grupos_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008';

## 3. Cobertura do filtro

Confirma o valor literal e o formato da empresa; não se trata de centro.

In [0]:
SELECT `cod_empresa` AS empresa_literal, LENGTH(TRIM(`cod_empresa`)) AS comprimento, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE TRIM(`cod_empresa`) RLIKE '^0*4008$' GROUP BY `cod_empresa`, LENGTH(TRIM(`cod_empresa`)) ORDER BY linhas DESC;

## 4. Granularidade

Compara fornecedores distintos com a chave fornecedor + empresa; não concatenar NULL.

In [0]:
WITH a AS (SELECT `cod_fornecedor`, `cod_empresa` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' GROUP BY `cod_fornecedor`, `cod_empresa`) SELECT (SELECT COUNT(*) FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008') AS linhas, (SELECT COUNT(*) FROM a) AS chaves_fornecedor_empresa, (SELECT COUNT(DISTINCT `cod_fornecedor`) FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008') AS fornecedores_distintos;

## 5. Duplicidade pura e versões

Hash interno da linha completa separa repetições idênticas de alterações; não expõe dados pessoais.

In [0]:
WITH h AS (SELECT t.`cod_fornecedor`, t.`cod_empresa`, MD5(TO_JSON(STRUCT(t.`cod_fornecedor`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_cnpj`, t.`cod_grupo_contas_fornecedor`, t.`ds_logradouro`, t.`nm_municipio`, t.`sg_estado`, t.`sg_pais`, t.`num_cep`, t.`num_telefone1`, t.`num_fax`, t.`cod_endereco`, t.`ind_excluido`, t.`ds_email`, t.`dh_carga`, t.`dateingest`, t.`yearingest`, t.`monthingest`), MAP('ignoreNullFields','false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` t WHERE `cod_empresa` = '4008'), d AS (SELECT `cod_fornecedor`, `cod_empresa`, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `cod_fornecedor`, `cod_empresa` HAVING COUNT(*)>1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas-versoes),0) AS repeticoes_identicas, COUNT_IF(versoes>1) AS chaves_com_versoes FROM d;

## 6. Preenchimento de todas as colunas

Perfil por coluna, incluindo campos pessoais apenas por contagem.

In [0]:
SELECT p.*, total-nulos-vazios AS preenchidos_nao_vazios, CASE WHEN total=0 THEN 'SEM LINHAS' WHEN nulos=total THEN '100% NULO: CONFERIR SAP' WHEN nulos+vazios=total THEN '100% SEM VALOR: CONFERIR SAP' ELSE 'COM DADOS' END AS veredito FROM (SELECT 'cod_fornecedor' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`cod_fornecedor` IS NULL) AS nulos, COUNT_IF(`cod_fornecedor` IS NOT NULL AND LOWER(TRIM(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'cod_empresa' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`cod_empresa` IS NULL) AS nulos, COUNT_IF(`cod_empresa` IS NOT NULL AND LOWER(TRIM(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'nm_fornecedor' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`nm_fornecedor` IS NULL) AS nulos, COUNT_IF(`nm_fornecedor` IS NOT NULL AND LOWER(TRIM(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'num_cnpj' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`num_cnpj` IS NULL) AS nulos, COUNT_IF(`num_cnpj` IS NOT NULL AND LOWER(TRIM(`num_cnpj`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_cnpj`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'cod_grupo_contas_fornecedor' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`cod_grupo_contas_fornecedor` IS NULL) AS nulos, COUNT_IF(`cod_grupo_contas_fornecedor` IS NOT NULL AND LOWER(TRIM(`cod_grupo_contas_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_grupo_contas_fornecedor`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'ds_logradouro' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`ds_logradouro` IS NULL) AS nulos, COUNT_IF(`ds_logradouro` IS NOT NULL AND LOWER(TRIM(`ds_logradouro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ds_logradouro`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'nm_municipio' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`nm_municipio` IS NULL) AS nulos, COUNT_IF(`nm_municipio` IS NOT NULL AND LOWER(TRIM(`nm_municipio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`nm_municipio`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'sg_estado' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`sg_estado` IS NULL) AS nulos, COUNT_IF(`sg_estado` IS NOT NULL AND LOWER(TRIM(`sg_estado`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`sg_estado`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'sg_pais' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`sg_pais` IS NULL) AS nulos, COUNT_IF(`sg_pais` IS NOT NULL AND LOWER(TRIM(`sg_pais`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`sg_pais`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'num_cep' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`num_cep` IS NULL) AS nulos, COUNT_IF(`num_cep` IS NOT NULL AND LOWER(TRIM(`num_cep`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_cep`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'num_telefone1' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`num_telefone1` IS NULL) AS nulos, COUNT_IF(`num_telefone1` IS NOT NULL AND LOWER(TRIM(`num_telefone1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_telefone1`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'num_fax' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`num_fax` IS NULL) AS nulos, COUNT_IF(`num_fax` IS NOT NULL AND LOWER(TRIM(`num_fax`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`num_fax`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'cod_endereco' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`cod_endereco` IS NULL) AS nulos, COUNT_IF(`cod_endereco` IS NOT NULL AND LOWER(TRIM(`cod_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`cod_endereco`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'ind_excluido' AS coluna, 'boolean' AS tipo, COUNT(*) AS total, COUNT_IF(`ind_excluido` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'ds_email' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`ds_email` IS NULL) AS nulos, COUNT_IF(`ds_email` IS NOT NULL AND LOWER(TRIM(`ds_email`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`ds_email`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'dh_carga' AS coluna, 'timestamp' AS tipo, COUNT(*) AS total, COUNT_IF(`dh_carga` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'dateingest' AS coluna, 'date' AS tipo, COUNT(*) AS total, COUNT_IF(`dateingest` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'yearingest' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`yearingest` IS NULL) AS nulos, COUNT_IF(`yearingest` IS NOT NULL AND LOWER(TRIM(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`yearingest`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' UNION ALL SELECT 'monthingest' AS coluna, 'string' AS tipo, COUNT(*) AS total, COUNT_IF(`monthingest` IS NULL) AS nulos, COUNT_IF(`monthingest` IS NOT NULL AND LOWER(TRIM(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS vazios, COUNT_IF(TRIM(`monthingest`) RLIKE '^0+([.,]0+)?$') AS zeros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008') p ORDER BY coluna;

## 6.1. Colunas sem fonte

O DESCRIBE não declara coluna física sem fonte; a seção 6 mede eventual ausência de preenchimento no recorte.

In [0]:
SELECT 'Sem coluna fisica declarada como sem fonte no DESCRIBE' AS observacao;

## 7. Cardinalidade

Confere categorias e identificadores sem publicar dados pessoais.

In [0]:
SELECT COUNT(DISTINCT `cod_fornecedor`) AS fornecedores, COUNT(DISTINCT `cod_grupo_contas_fornecedor`) AS grupos_contas, COUNT(DISTINCT `cod_endereco`) AS enderecos_tecnicos, COUNT(DISTINCT `sg_estado`) AS estados FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008';

## 8. Domínio

Oito grupos de contas mais comuns para eventual segundo cenário, sem aplicar filtro no primeiro.

In [0]:
SELECT `cod_grupo_contas_fornecedor`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' GROUP BY `cod_grupo_contas_fornecedor` ORDER BY linhas DESC, `cod_grupo_contas_fornecedor` LIMIT 8;

## 9. Indicadores

Separar fornecedores marcados para exclusão sem excluir registros deste cenário.

In [0]:
SELECT `ind_excluido`, COUNT(*) AS linhas, COUNT_IF(`ds_email` IS NULL OR TRIM(`ds_email`)='') AS sem_email FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' GROUP BY `ind_excluido` ORDER BY linhas DESC;

## 10. Perfil numérico

O DESCRIBE não traz valores monetários ou quantidades numéricas; códigos não são medidas.

In [0]:
SELECT 'Sem medidas numericas no DESCRIBE' AS observacao;

## 11. Totais para conciliação

Não somar identificadores; o volume e a chave são as medidas relevantes.

In [0]:
SELECT COUNT(*) AS total_linhas, COUNT(DISTINCT `cod_fornecedor`) AS fornecedores FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008';

## 12. Datas

Mede o carimbo de carga e a data de ingestão, sem interpretá-los como período do SAP.

In [0]:
SELECT MIN(`dh_carga`) AS menor_carga, MAX(`dh_carga`) AS maior_carga, MIN(`dateingest`) AS primeira_ingestao, MAX(`dateingest`) AS ultima_ingestao FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008';

## 13. Códigos e zeros à esquerda

Confere a normalização do código do fornecedor para comparação SAP.

In [0]:
SELECT COUNT(*) AS linhas, COUNT_IF(TRIM(`cod_fornecedor`) RLIKE '^0+[0-9]+$') AS fornecedores_com_zeros, COUNT(DISTINCT CASE WHEN TRIM(`cod_fornecedor`) RLIKE '^[0-9]+$' THEN TRIM(`cod_fornecedor`) END) AS distintos_numericos, COUNT(DISTINCT CASE WHEN TRIM(`cod_fornecedor`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_fornecedor`),'^0+(?=[0-9])','') END) AS distintos_normalizados FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008';

## 14. Checksum

Mede duplicatas 100% idênticas por hash sem revelar registros.

In [0]:
WITH h AS (SELECT MD5(TO_JSON(STRUCT(t.`cod_fornecedor`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_cnpj`, t.`cod_grupo_contas_fornecedor`, t.`ds_logradouro`, t.`nm_municipio`, t.`sg_estado`, t.`sg_pais`, t.`num_cep`, t.`num_telefone1`, t.`num_fax`, t.`cod_endereco`, t.`ind_excluido`, t.`ds_email`, t.`dh_carga`, t.`dateingest`, t.`yearingest`, t.`monthingest`), MAP('ignoreNullFields','false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` t WHERE `cod_empresa` = '4008'), d AS (SELECT hash_linha, COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas, COUNT(*) AS hashes_distintos, COALESCE(SUM(linhas-1),0) AS repeticoes_identicas FROM d;

## 15. Amostra

Até 20 linhas sem valores pessoais literais.

In [0]:
SELECT `cod_fornecedor`, `cod_empresa`, `cod_grupo_contas_fornecedor`, `sg_estado`, `sg_pais`, `cod_endereco`, `ind_excluido`, `dh_carga`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`num_cnpj` AS STRING)),1,8) AS `num_cnpj_hash`, SUBSTRING(MD5(CAST(`ds_logradouro` AS STRING)),1,8) AS `ds_logradouro_hash`, SUBSTRING(MD5(CAST(`nm_municipio` AS STRING)),1,8) AS `nm_municipio_hash`, SUBSTRING(MD5(CAST(`num_cep` AS STRING)),1,8) AS `num_cep_hash`, SUBSTRING(MD5(CAST(`num_telefone1` AS STRING)),1,8) AS `num_telefone1_hash`, SUBSTRING(MD5(CAST(`num_fax` AS STRING)),1,8) AS `num_fax_hash`, SUBSTRING(MD5(CAST(`ds_email` AS STRING)),1,8) AS `ds_email_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' ORDER BY `cod_fornecedor` LIMIT 20;

## 16. Distribuição interna

Perfis por grupo de contas, UF e indicador de exclusão, sem filtrar o universo.

In [0]:
SELECT `cod_grupo_contas_fornecedor`, `sg_estado`, `ind_excluido`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' GROUP BY `cod_grupo_contas_fornecedor`, `sg_estado`, `ind_excluido` ORDER BY linhas DESC LIMIT 100;

## 17. Freshness

Diferencia carimbo raw e partição curated; a data de ingestão não é filtro SAP.

In [0]:
SELECT `dateingest`, MIN(`dh_carga`) AS menor_carga_raw, MAX(`dh_carga`) AS maior_carga_raw, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' GROUP BY `dateingest` ORDER BY `dateingest` DESC;

## 18. Regras da ZGLMM214

Mede a cobertura de e-mail por grupo sem excluir fornecedores sem e-mail.

In [0]:
SELECT `cod_grupo_contas_fornecedor`, COUNT(*) AS linhas, COUNT_IF(`ds_email` IS NULL OR TRIM(`ds_email`)='') AS sem_email, COUNT_IF(`ds_email` IS NOT NULL AND TRIM(`ds_email`)<>'') AS com_email, COUNT_IF(`ind_excluido`=TRUE) AS excluidos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' GROUP BY `cod_grupo_contas_fornecedor` ORDER BY linhas DESC;

## 18.1. E-mail por endereço técnico

Mede endereços com mais de uma versão de e-mail sem publicar o endereço eletrônico.

In [0]:
WITH a AS (SELECT `cod_endereco`, COUNT(*) AS linhas, COUNT(DISTINCT `ds_email`) AS emails_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' GROUP BY `cod_endereco`) SELECT COUNT(*) AS enderecos, COUNT_IF(linhas>1) AS enderecos_em_mais_de_um_registro, COUNT_IF(emails_distintos>1) AS enderecos_com_emails_diferentes FROM a;

## 19. Amostra ampliada

Até 500 linhas, mesmas regras de proteção de dados da amostra.

In [0]:
SELECT `cod_fornecedor`, `cod_empresa`, `cod_grupo_contas_fornecedor`, `sg_estado`, `sg_pais`, `cod_endereco`, `ind_excluido`, `dh_carga`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`num_cnpj` AS STRING)),1,8) AS `num_cnpj_hash`, SUBSTRING(MD5(CAST(`ds_logradouro` AS STRING)),1,8) AS `ds_logradouro_hash`, SUBSTRING(MD5(CAST(`nm_municipio` AS STRING)),1,8) AS `nm_municipio_hash`, SUBSTRING(MD5(CAST(`num_cep` AS STRING)),1,8) AS `num_cep_hash`, SUBSTRING(MD5(CAST(`num_telefone1` AS STRING)),1,8) AS `num_telefone1_hash`, SUBSTRING(MD5(CAST(`num_fax` AS STRING)),1,8) AS `num_fax_hash`, SUBSTRING(MD5(CAST(`ds_email` AS STRING)),1,8) AS `ds_email_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' ORDER BY `cod_fornecedor` LIMIT 500;

## 20. Resumo

Bloco para conciliar volume, chave e perfil; esperado é valor histórico, não garantia.

In [0]:
WITH d AS (SELECT `cod_fornecedor`,`cod_empresa`,COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' GROUP BY `cod_fornecedor`,`cod_empresa`) SELECT 'ZGLMM214' AS transacao, '4008' AS empresa, 183266 AS volume_esperado_exploratoria, (SELECT COUNT(*) FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008') AS volume_atual, (SELECT COUNT(*) FROM d) AS chaves_distintas, (SELECT COUNT(*) FROM d WHERE linhas>1) AS chaves_repetidas, 'cod_fornecedor + cod_empresa' AS chave_sugerida;

## 21. EXTRAÇÃO COMPLETA para comparação

Sem LIMIT; todos os fornecedores e grupos da empresa 4008, inclusive excluídos e sem e-mail. Campos pessoais saem somente como hash.

In [0]:
SELECT CASE WHEN TRIM(`cod_fornecedor`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_fornecedor`),'^0+(?=[0-9])','') ELSE TRIM(`cod_fornecedor`) END AS `cod_fornecedor_normalizado`, `cod_fornecedor`, `cod_empresa`, `cod_grupo_contas_fornecedor`, `sg_estado`, `sg_pais`, `cod_endereco`, `ind_excluido`, `dh_carga`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`num_cnpj` AS STRING)),1,8) AS `num_cnpj_hash`, SUBSTRING(MD5(CAST(`ds_logradouro` AS STRING)),1,8) AS `ds_logradouro_hash`, SUBSTRING(MD5(CAST(`nm_municipio` AS STRING)),1,8) AS `nm_municipio_hash`, SUBSTRING(MD5(CAST(`num_cep` AS STRING)),1,8) AS `num_cep_hash`, SUBSTRING(MD5(CAST(`num_telefone1` AS STRING)),1,8) AS `num_telefone1_hash`, SUBSTRING(MD5(CAST(`num_fax` AS STRING)),1,8) AS `num_fax_hash`, SUBSTRING(MD5(CAST(`ds_email` AS STRING)),1,8) AS `ds_email_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` WHERE `cod_empresa` = '4008' ORDER BY `cod_fornecedor`, `cod_empresa`;

## Checklist SAP

- [ ] Na tela SAP, informar **Empresa 4008**; deixar **Fornecedor** e **Grupo de contas** em branco. Layout é apenas apresentação.
- [ ] Confirmar no SAP se o relatório inclui fornecedores excluídos e fornecedores sem e-mail; se não, medir a diferença e definir filtros equivalentes antes de concluir.
- [ ] Executar as extrações SAP e DL no mesmo dia e registrar horário/snapshot.
- [ ] Conferir volumes, chave fornecedor + empresa, duplicatas e zeros à esquerda.
- [ ] Conferir ausência de campos não disponíveis no Datalake e tratar dados pessoais somente em ambiente autorizado; não exportar valores literais.
- [ ] O nome `Centro4008` é somente convenção do arquivo. O cenário **não** tem filtro de centro.